In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata
from sklearn.preprocessing import MinMaxScaler

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Skin', 'Inflamed', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/rna/raw/intersect_atac.h5ad')

In [ ]:
adata

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/csv/clusters/rna_leiden_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t',index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
adata.obs['coarse'] = clusters.reindex(adata.obs.index)['coarse_v7']

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
degs_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/csv/degs/justin_degs_v7.csv'
degs = pd.read_csv(degs_path, sep='\t')
degs.shape

In [ ]:
degs.head()

In [ ]:
adata.var.head()

In [ ]:
plot_degs = []
for clust in tmp_order:
    tmp_degs = degs[degs['coarse_v7'] == clust].sort_values(by=['avg_log2FC', 'p_val_adj'], ascending=False)
    print(clust,tmp_degs['gene'][:5].values)
    plot_degs.append(tmp_degs['gene'][:5].values)
    
plot_degs = pd.Index(np.ravel(plot_degs))
mask = plot_degs.isin(adata.var.index)
plot_degs = plot_degs[mask]

indices = []
for g in plot_degs:
    idx = np.where(adata.var.index == g)[0][0]
    indices.append(idx)
    
final_adata = adata[:,indices]

In [ ]:
sc.pl.dotplot(final_adata, var_names=final_adata.var.index, groupby='coarse', vmax=2)

In [ ]:
degs_df = pd.DataFrame(final_adata.X.toarray())
degs_df.index = adata.obs.index
degs_df.columns = final_adata.var.index
degs_df.shape

In [ ]:
degs_df.head()

### hm these genes aren't very good. let's just pick a few manually

In [ ]:
#               'SOX2', 'KLF4', 'POU5F1',# Sendai
#               'KLK14', 'IGF2', 'LYPD2', # Skin
#               'PMAIP1', 'IFIT2', 'HIP1R', # Inflamed
#               'RFX8', 'ANPEP', 'STC1', # Delayed
#                'PRRX1', 'IFI6', 'IFI27', # Fail1
#                'RPL19', 'RPS12', 'EEF1A1', # Inter1
#                'TENM1', 'SYNE2', 'TBCD', # Inter2
#                'GABRA2', 'NKAIN3', 'SNTG1', # Fail2
#                'GABRB3', 'CHRM3', 'DSG2' # IPS
#               ]

In [ ]:
select_degs = ['CEBPD', 'THY1', 'CXCL12', # Start
              'SOX2', 'KLF4', 'POU5F1',# Sendai
              'KLK14', 'IGF2', 'LYPD2', # Skin
              'PMAIP1', 'IFIT2', 'HIP1R', # Inflamed
              'HMGA1', 'PODXL', 'IL6', # Delayed
               'COL11A1', 'NUAK1', 'RUNX1', # Fail1
               'NFIA', 'CALB1', # Inter1
               'XACT', 'LEFTY2', 'TENM1', # Inter2
               'COL19A1', 'SNTG1', # Fail2
               'LIN28B', 'DNMT3B', 'TET1', 'EPCAM', # IPS
              ]

select_degs = pd.Index(select_degs)
mask = select_degs.isin(adata.var.index)
final_degs = select_degs[mask]
final_degs.shape

In [ ]:
indices = []
for g in final_degs:
    idx = np.where(adata.var.index == g)[0][0]
    indices.append(idx)
    
final_adata = adata[:,indices]
final_adata

In [ ]:
sc.pl.dotplot(final_adata, var_names=final_adata.var.index, groupby='coarse', vmax=2)

In [ ]:
pd.Index(np.ravel(final_adata.X[:,0].toarray())).value_counts()

In [ ]:
degs_df = pd.DataFrame(final_adata.X.toarray())
degs_df.index = adata.obs.index
degs_df.columns = final_adata.var.index
degs_df.shape

In [ ]:
tmp_meta = adata.obs[['coarse']]
tmp_meta.head()

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure1'
degs_df.to_csv(f'{indir}/multiome_dot_plot.csv', sep='\t')
tmp_meta.to_csv(f'{indir}/multiome_dot_plot_meta.csv', sep='\t')

In [ ]:
!date